# 00 Data Exploration — M5 Retail Demand

**Goal:** Build a clean, defensible exploratory analysis of the M5 dataset before modeling.

This notebook focuses on:

1. Dataset structure and data quality
2. Retail hierarchy and aggregation levels
3. Daily sales trends by category, store, and state
4. Extreme calendar/event behavior such as Christmas and Thanksgiving
5. Sales distribution analysis
6. Seasonality, SNAP/event effects, and price behavior
7. SKU-store intermittency profiles
8. Simple 28-day baseline forecasting checks

**Important modeling principle:** keep the raw data intact, create event flags, and only exclude extreme days in diagnostic views when the goal is to understand normal demand behavior.

## 1. Setup

This notebook keeps the setup block intentionally thin. Reusable plotting, path resolution, and calendar helpers live in `helpers/eda.py` so they can be reused in later feature engineering notebooks.


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SHARED_ROOT = None
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "helpers" / "__init__.py").exists():
        SHARED_ROOT = candidate
        break

if SHARED_ROOT is None:
    raise FileNotFoundError("Could not locate the shared portfolio helpers directory.")

if str(SHARED_ROOT) not in sys.path:
    sys.path.insert(0, str(SHARED_ROOT))

from helpers import (
    add_calendar_flags,
    bootstrap_notebook,
    finish_axis,
    plot_category_bar_panels,
    plot_category_histograms,
    plot_category_multiline,
    plot_category_panels,
)

notebook_context = bootstrap_notebook(Path.cwd())
PROJECT_ROOT = notebook_context.project_root
DATA_DIR = notebook_context.data_dir
PROCESSED_DIR = notebook_context.processed_dir
category_colors = notebook_context.category_colors


ModuleNotFoundError: No module named 'numpy'

## 2. Load M5 data

`datasetsforecast` returns three tables:

- `Y_df`: time-series target sales
- `X_df`: time-varying exogenous variables such as price, events, and SNAP flags
- `S_df`: static hierarchy variables such as item, department, category, store, and state

In [ ]:
from datasetsforecast.m5 import M5

Y_df, X_df, S_df = M5.load(directory=DATA_DIR, cache=True)

print("Y_df:", Y_df.shape)
print("X_df:", X_df.shape)
print("S_df:", S_df.shape)

In [ ]:
Y_df.head()

In [ ]:
X_df.head()

In [ ]:
S_df.head()

## 3. Initial schema and data quality audit

In [ ]:
for name, df in {"Y_df": Y_df, "X_df": X_df, "S_df": S_df}.items():
    print(f"\n{name}")
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())
    print(df.dtypes)

In [ ]:
print("Date range:", Y_df["ds"].min(), "to", Y_df["ds"].max())
print("Number of unique series:", Y_df["unique_id"].nunique())
print("Rows:", len(Y_df))
print("Duplicate unique_id-date rows:", Y_df.duplicated(subset=["unique_id", "ds"]).sum())
print("Negative sales rows:", (Y_df["y"] < 0).sum())
print("Missing target values:", Y_df["y"].isna().sum())

## 4. Create the main joined table

This table joins sales, time-varying features, and static hierarchy into one SKU-store-day table.

In [ ]:
m5 = (
    Y_df
    .merge(X_df, on=["unique_id", "ds"], how="left")
    .merge(S_df, on="unique_id", how="left")
)

print("m5 shape:", m5.shape)
m5.head()

In [ ]:
missing_summary = (
    m5.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .reset_index()
    .rename(columns={"index": "column"})
)

missing_summary.head(25)

In [ ]:
series_lengths = (
    m5.groupby("unique_id", observed=True)["ds"]
    .agg(start_date="min", end_date="max", observations="count")
    .reset_index()
)

series_lengths.describe(include="all")

## 5. Hierarchy exploration

The goal here is to understand the retail structure before forecasting: states, stores, categories, departments, and items.

In [ ]:
hierarchy_cols = ["state_id", "store_id", "cat_id", "dept_id", "item_id"]

S_df[hierarchy_cols].nunique()

In [ ]:
unique_values = {
    col: sorted(S_df[col].dropna().unique().tolist())
    for col in hierarchy_cols
}

# item_id has many values, so preview only the first 20
for col, values in unique_values.items():
    print(f"\n{col}: {len(values)} unique")
    print(values[:20])

In [ ]:
store_category_structure = (
    S_df
    .groupby(["state_id", "store_id", "cat_id"], observed=True)
    .agg(
        item_count=("item_id", "nunique"),
        series_count=("unique_id", "nunique"),
    )
    .reset_index()
    .sort_values(["state_id", "store_id", "cat_id"])
)

store_category_structure

In [ ]:
hierarchy_summary = (
    S_df
    .groupby(["state_id", "store_id", "cat_id", "dept_id"], observed=True)
    .agg(series_count=("unique_id", "nunique"))
    .reset_index()
    .sort_values(["state_id", "store_id", "cat_id", "dept_id"])
)

hierarchy_summary.head(25)

## 6. Build core aggregate tables

We keep three important views:

1. `category_daily_raw`: all dates retained, source of truth for category-level EDA
2. `category_daily_event_adjusted`: same data with event/holiday flags added
3. `category_daily_normalized`: diagnostic view excluding Christmas only, useful for normal demand distribution checks

Do **not** treat the normalized table as the source of truth. It is only for specific diagnostics.

In [ ]:
category_daily_raw = (
    m5
    .groupby(["ds", "cat_id"], observed=True)
    .agg(y=("y", "sum"))
    .reset_index()
    .sort_values(["cat_id", "ds"])
)

category_daily_raw.head()

In [ ]:
category_daily_events = (
    m5
    .groupby(["ds", "cat_id"], observed=True)
    .agg(
        y=("y", "sum"),
        event_name_1=("event_name_1", "first"),
        event_type_1=("event_type_1", "first"),
        event_name_2=("event_name_2", "first"),
        event_type_2=("event_type_2", "first"),
    )
    .reset_index()
    .sort_values(["cat_id", "ds"])
)

category_daily_events.head()

In [ ]:
def add_calendar_flags(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["year"] = out["ds"].dt.year
    out["month"] = out["ds"].dt.month
    out["day"] = out["ds"].dt.day
    out["dayofweek"] = out["ds"].dt.day_name()
    out["weekofyear"] = out["ds"].dt.isocalendar().week.astype(int)
    out["is_christmas"] = (out["month"].eq(12) & out["day"].eq(25))
    out["is_christmas_eve"] = (out["month"].eq(12) & out["day"].eq(24))
    out["is_day_after_christmas"] = (out["month"].eq(12) & out["day"].eq(26))
    out["is_new_years_day"] = (out["month"].eq(1) & out["day"].eq(1))
    out["is_thanksgiving_window"] = (out["month"].eq(11) & out["day"].between(22, 28))
    return out

category_daily_event_adjusted = add_calendar_flags(category_daily_events)

category_daily_normalized = category_daily_event_adjusted.loc[
    ~category_daily_event_adjusted["is_christmas"]
].copy()

print("Raw rows:", len(category_daily_event_adjusted))
print("Normalized rows:", len(category_daily_normalized))

In [ ]:
store_category_daily = (
    m5
    .groupby(["ds", "state_id", "store_id", "cat_id"], observed=True)
    .agg(y=("y", "sum"))
    .reset_index()
    .sort_values(["state_id", "store_id", "cat_id", "ds"])
)

state_category_daily = (
    m5
    .groupby(["ds", "state_id", "cat_id"], observed=True)
    .agg(y=("y", "sum"))
    .reset_index()
    .sort_values(["state_id", "cat_id", "ds"])
)

store_category_daily.head()

## 7. Total and category-level sales trends

In [ ]:
daily_sales = (
    category_daily_raw
    .groupby("ds", observed=True)
    .agg(y=("y", "sum"))
    .reset_index()
)

ax = daily_sales.plot(x="ds", y="y", figsize=(16, 6), legend=False)
ax.set_title("Total Daily Unit Sales")
ax.set_xlabel("Date")
ax.set_ylabel("Units Sold")
ax.grid(False)
plt.show()

In [ ]:
category_pivot = category_daily_raw.pivot(index="ds", columns="cat_id", values="y")

ax = category_pivot.plot(
    figsize=(16, 7),
    color=[category_colors.get(col, "tab:gray") for col in category_pivot.columns],
)
ax.set_title("Daily Sales by Category | Raw")
ax.set_xlabel("Date")
ax.set_ylabel("Units Sold")
ax.grid(False)
plt.show()

In [ ]:
for cat in sorted(category_daily_raw["cat_id"].unique()):
    data = category_daily_raw.loc[category_daily_raw["cat_id"].eq(cat)].sort_values("ds")

    ax = data.plot(
        x="ds",
        y="y",
        figsize=(16, 6),
        title=f"Daily Sales - {cat} | Raw",
        legend=False,
        color=category_colors.get(cat, "tab:gray"),
    )
    ax.set_xlabel("Date")
    ax.set_ylabel("Units Sold")
    ax.grid(False)
    plt.show()

In [ ]:
category_pivot_norm = category_daily_normalized.pivot(index="ds", columns="cat_id", values="y")

ax = category_pivot_norm.plot(
    figsize=(16, 7),
    color=[category_colors.get(col, "tab:gray") for col in category_pivot_norm.columns],
)
ax.set_title("Daily Sales by Category | Excluding Christmas Diagnostic View")
ax.set_xlabel("Date")
ax.set_ylabel("Units Sold")
ax.grid(False)
plt.show()

## 8. Extreme low-sales days and event diagnostics

This section investigates whether low observations are true low demand, event-driven behavior, store-level closure-like behavior, or data issues.

In [ ]:
zero_category_days = category_daily_raw.loc[category_daily_raw["y"].eq(0)].copy()

print("Zero category-day observations:", len(zero_category_days))
zero_category_days.sort_values(["ds", "cat_id"])

In [ ]:
low_sales_days = (
    category_daily_event_adjusted
    .assign(
        p01_by_category=lambda df: df.groupby("cat_id", observed=True)["y"].transform(lambda s: s.quantile(0.01))
    )
    .loc[lambda df: df["y"] <= df["p01_by_category"]]
    .sort_values(["cat_id", "y"])
)

low_sales_days

In [ ]:
# Christmas impact: how much sales are retained vs ordinary days?
christmas_impact = (
    category_daily_event_adjusted
    .groupby(["cat_id", "is_christmas"], observed=True)["y"]
    .agg(count="count", mean="mean", median="median", min="min", max="max")
    .reset_index()
)

christmas_impact_pivot = (
    christmas_impact
    .pivot(index="cat_id", columns="is_christmas", values="mean")
    .rename(columns={False: "normal_avg_sales", True: "christmas_avg_sales"})
)

christmas_impact_pivot["christmas_sales_retention"] = (
    christmas_impact_pivot["christmas_avg_sales"] / christmas_impact_pivot["normal_avg_sales"]
)
christmas_impact_pivot["christmas_sales_drop_pct"] = 1 - christmas_impact_pivot["christmas_sales_retention"]

christmas_impact_pivot.style.format({
    "normal_avg_sales": "{:,.0f}",
    "christmas_avg_sales": "{:,.0f}",
    "christmas_sales_retention": "{:.2%}",
    "christmas_sales_drop_pct": "{:.2%}",
})

In [ ]:
# Drill down into Christmas sales by store/category/item.
christmas_detail = m5.loc[
    (m5["ds"].dt.month.eq(12))
    & (m5["ds"].dt.day.eq(25))
    & (m5["y"] > 0)
].copy()

christmas_store_sales = (
    christmas_detail
    .assign(year=lambda df: df["ds"].dt.year)
    .groupby(["year", "cat_id", "state_id", "store_id"], observed=True)
    .agg(
        total_sales=("y", "sum"),
        selling_items=("item_id", "nunique"),
        selling_series=("unique_id", "nunique"),
    )
    .reset_index()
    .sort_values(["year", "cat_id", "total_sales"], ascending=[True, True, False])
)

christmas_store_sales

In [ ]:
christmas_category_summary = (
    christmas_store_sales
    .groupby(["year", "cat_id"], observed=True)
    .agg(
        stores_with_sales=("store_id", "nunique"),
        total_christmas_sales=("total_sales", "sum"),
        total_selling_items=("selling_items", "sum"),
        total_selling_series=("selling_series", "sum"),
    )
    .reset_index()
)

christmas_category_summary

In [ ]:
# Thanksgiving is another visible low-demand window for some categories.
thanksgiving_impact = (
    category_daily_event_adjusted
    .groupby(["cat_id", "is_thanksgiving_window"], observed=True)["y"]
    .agg(count="count", mean="mean", median="median", min="min", max="max")
    .reset_index()
)

thanksgiving_impact

### Working interpretation

Christmas should not automatically be removed as a data error. It appears to be a predictable extreme calendar event where sales drop sharply, sometimes near zero. For diagnostic views, excluding Christmas can help reveal normal demand behavior. For forecasting, Christmas should be retained with event flags so models can learn the expected demand suppression.

## 9. Sales distribution analysis

Use both raw and normalized views. The raw view shows real operating behavior. The normalized view helps study normal category demand without Christmas dominating the distribution.

In [ ]:
category_summary_raw = (
    category_daily_raw
    .groupby("cat_id", observed=True)["y"]
    .agg(count="count", mean="mean", median="median", std="std", skew="skew", min="min", max="max")
    .round(2)
)

category_summary_raw

In [ ]:
category_summary_normalized = (
    category_daily_normalized
    .groupby("cat_id", observed=True)["y"]
    .agg(count="count", mean="mean", median="median", std="std", skew="skew", min="min", max="max")
    .round(2)
)

category_summary_normalized

In [ ]:
def plot_category_histograms(df: pd.DataFrame, title_suffix: str, density: bool = False) -> None:
    y_label = "Density" if density else "Frequency"
    plot_type = "Density" if density else "Frequency"

    for cat in sorted(df["cat_id"].unique()):
        data = df.loc[df["cat_id"].eq(cat), "y"]
        mean_sales = data.mean()
        median_sales = data.median()

        ax = data.hist(
            bins="sqrt",
            density=density,
            figsize=(12, 4),
            color=category_colors.get(cat, "tab:gray"),
            edgecolor="black",
            alpha=0.45,
        )

        ax.axvline(
            mean_sales,
            color="red",
            linestyle="--",
            linewidth=2,
            label=f"Mean: {mean_sales:,.0f}",
        )
        ax.axvline(
            median_sales,
            color="black",
            linestyle="-",
            linewidth=2,
            label=f"Median: {median_sales:,.0f}",
        )

        ax.set_title(f"Daily Sales {plot_type} Distribution - {cat} | {title_suffix}")
        ax.set_xlabel("Daily Units Sold")
        ax.set_ylabel(y_label)
        ax.legend()
        ax.grid(False)
        plt.show()

In [ ]:
plot_category_histograms(category_daily_normalized, title_suffix="Excluding Christmas", density=False)

In [ ]:
plot_category_histograms(category_daily_normalized, title_suffix="Excluding Christmas", density=True)

In [ ]:
# Combined density plot with overall mean and median
overall_mean = category_daily_normalized["y"].mean()
overall_median = category_daily_normalized["y"].median()

plt.figure(figsize=(14, 6))

for cat in sorted(category_daily_normalized["cat_id"].unique()):
    data = category_daily_normalized.loc[category_daily_normalized["cat_id"].eq(cat), "y"]
    plt.hist(
        data,
        bins="sqrt",
        density=True,
        alpha=0.45,
        edgecolor="black",
        label=cat,
        color=category_colors.get(cat, "tab:gray"),
    )

plt.axvline(
    overall_mean,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Overall Mean: {overall_mean:,.0f}",
)
plt.axvline(
    overall_median,
    color="black",
    linestyle="-",
    linewidth=2,
    label=f"Overall Median: {overall_median:,.0f}",
)

plt.title("Daily Sales Distribution by Category | Density with Overall Mean and Median")
plt.xlabel("Daily Units Sold")
plt.ylabel("Density")
plt.legend()
plt.grid(False)
plt.show()

### Distribution interpretation

The combined distribution can appear right-skewed, with the overall mean above the overall median. However, the more important finding is that the categories operate on materially different sales scales. `HOBBIES`, `HOUSEHOLD`, and `FOODS` form distinct clusters, making global mean and median poor representatives of category-level demand behavior. Because of this, distribution analysis, seasonality checks, and forecasting diagnostics should be performed by category.

## 10. Seasonality diagnostics

Analyze seasonality by category instead of only globally. This avoids allowing high-volume categories to dominate the interpretation.

In [ ]:
seasonality_df = add_calendar_flags(category_daily_raw)
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

category_dow_sales = (
    seasonality_df
    .groupby(["cat_id", "dayofweek"], observed=True)["y"]
    .mean()
    .reset_index()
)
category_dow_sales["dayofweek"] = pd.Categorical(category_dow_sales["dayofweek"], categories=weekday_order, ordered=True)
category_dow_sales = category_dow_sales.sort_values(["cat_id", "dayofweek"])

category_dow_sales

In [ ]:
for cat in sorted(category_dow_sales["cat_id"].unique()):
    data = category_dow_sales.loc[category_dow_sales["cat_id"].eq(cat)]
    ax = data.plot(
        x="dayofweek",
        y="y",
        kind="bar",
        figsize=(10, 4),
        legend=False,
        color=category_colors.get(cat, "tab:gray"),
        edgecolor="black",
        alpha=0.75,
        title=f"Average Daily Sales by Day of Week - {cat}",
    )
    ax.set_xlabel("Day of Week")
    ax.set_ylabel("Average Units Sold")
    ax.grid(False)
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
category_monthly_sales = (
    seasonality_df
    .groupby(["cat_id", "month"], observed=True)["y"]
    .mean()
    .reset_index()
)

for cat in sorted(category_monthly_sales["cat_id"].unique()):
    data = category_monthly_sales.loc[category_monthly_sales["cat_id"].eq(cat)]
    ax = data.plot(
        x="month",
        y="y",
        kind="bar",
        figsize=(10, 4),
        legend=False,
        color=category_colors.get(cat, "tab:gray"),
        edgecolor="black",
        alpha=0.75,
        title=f"Average Daily Sales by Month - {cat}",
    )
    ax.set_xlabel("Month")
    ax.set_ylabel("Average Units Sold")
    ax.grid(False)
    plt.show()

## 11. Event and SNAP effects

In [ ]:
event_sales = (
    m5
    .groupby(["cat_id", "event_type_1"], observed=True)["y"]
    .agg(mean="mean", median="median", count="count")
    .reset_index()
    .sort_values(["cat_id", "mean"], ascending=[True, False])
)

event_sales

In [ ]:
snap_cols = ["snap_CA", "snap_TX", "snap_WI"]

snap_effects = []
for state, snap_col in zip(["CA", "TX", "WI"], snap_cols):
    tmp = (
        m5.loc[m5["state_id"].eq(state)]
        .groupby(["cat_id", snap_col], observed=True)["y"]
        .agg(mean="mean", median="median", count="count")
        .reset_index()
        .rename(columns={snap_col: "snap_flag"})
    )
    tmp["state_id"] = state
    tmp["snap_col"] = snap_col
    snap_effects.append(tmp)

snap_effects = pd.concat(snap_effects, ignore_index=True)
snap_effects.sort_values(["state_id", "cat_id", "snap_flag"])

## 12. SKU-store intermittency and demand profile

This is critical for supply-chain logic because many SKU-store series are sparse or intermittent even when category-level demand looks smooth.

In [ ]:
series_profile = (
    m5
    .groupby("unique_id", observed=True)
    .agg(
        avg_daily_sales=("y", "mean"),
        median_daily_sales=("y", "median"),
        total_sales=("y", "sum"),
        zero_sales_rate=("y", lambda s: (s == 0).mean()),
        sales_std=("y", "std"),
        days_observed=("y", "count"),
    )
    .reset_index()
    .merge(S_df, on="unique_id", how="left")
)

series_profile.head()

In [ ]:
def demand_type(row):
    if row["zero_sales_rate"] >= 0.70:
        return "Very intermittent"
    if row["zero_sales_rate"] >= 0.40:
        return "Intermittent"
    if row["avg_daily_sales"] >= 5:
        return "High volume"
    return "Regular / low volume"

series_profile["demand_type"] = series_profile.apply(demand_type, axis=1)

series_profile["demand_type"].value_counts()

In [ ]:
series_profile_by_category = (
    series_profile
    .groupby(["cat_id", "demand_type"], observed=True)
    .agg(
        series_count=("unique_id", "nunique"),
        avg_zero_rate=("zero_sales_rate", "mean"),
        avg_daily_sales=("avg_daily_sales", "mean"),
    )
    .reset_index()
    .sort_values(["cat_id", "series_count"], ascending=[True, False])
)

series_profile_by_category

## 13. Price behavior

In [ ]:
price_profile = (
    m5
    .groupby("unique_id", observed=True)
    .agg(
        avg_price=("sell_price", "mean"),
        min_price=("sell_price", "min"),
        max_price=("sell_price", "max"),
        price_changes=("sell_price", "nunique"),
        total_sales=("y", "sum"),
    )
    .reset_index()
    .merge(S_df, on="unique_id", how="left")
)

price_profile.sort_values("price_changes", ascending=False).head(20)

In [ ]:
# Inspect one high-volume series with price movement.
sample_id = series_profile.sort_values("total_sales", ascending=False)["unique_id"].iloc[0]
sample = m5.loc[m5["unique_id"].eq(sample_id)].copy().sort_values("ds")

ax = sample.plot(
    x="ds",
    y=["y", "sell_price"],
    figsize=(16, 6),
    title=f"Sales and Price - {sample_id}",
)
ax.set_xlabel("Date")
ax.grid(False)
plt.show()

## 14. Simple 28-day baseline forecasting

Before advanced models, create simple baselines. The goal is not to win accuracy yet; it is to understand benchmark difficulty and behavior by category.

In [ ]:
H = 28

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def wape(y_true, y_pred):
    denom = np.sum(np.abs(y_true))
    return np.nan if denom == 0 else np.sum(np.abs(y_true - y_pred)) / denom

def evaluate_baselines_one_series(df: pd.DataFrame, h: int = 28) -> dict:
    df = df.sort_values("ds")
    train = df.iloc[:-h].copy()
    test = df.iloc[-h:].copy()

    forecasts = {
        "naive_last": train["y"].iloc[-1],
        "ma_28": train["y"].tail(28).mean(),
        "ma_56": train["y"].tail(56).mean(),
        "same_weekday_4wk": train["y"].tail(28).mean(),  # placeholder, replace later with weekday-specific logic
    }

    rows = []
    for model_name, pred in forecasts.items():
        rows.append({
            "model": model_name,
            "mae": mae(test["y"], pred),
            "wape": wape(test["y"], pred),
        })
    return rows

In [ ]:
# Category-level baseline forecast evaluation
baseline_results = []

for cat in sorted(category_daily_raw["cat_id"].unique()):
    df = category_daily_raw.loc[category_daily_raw["cat_id"].eq(cat), ["ds", "y"]].copy()
    for row in evaluate_baselines_one_series(df, h=H):
        row["cat_id"] = cat
        baseline_results.append(row)

baseline_results = pd.DataFrame(baseline_results)
baseline_results.sort_values(["cat_id", "wape"])

In [ ]:
# Visualize the final 28 days for one category.
cat = "FOODS"
df = category_daily_raw.loc[category_daily_raw["cat_id"].eq(cat), ["ds", "y"]].sort_values("ds").copy()
train = df.iloc[:-H].copy()
test = df.iloc[-H:].copy()

last_value = train["y"].iloc[-1]
ma_28 = train["y"].tail(28).mean()
ma_56 = train["y"].tail(56).mean()

plot_df = test.copy()
plot_df["naive_last"] = last_value
plot_df["ma_28"] = ma_28
plot_df["ma_56"] = ma_56

ax = plot_df.plot(x="ds", y=["y", "naive_last", "ma_28", "ma_56"], figsize=(14, 5))
ax.set_title(f"28-Day Baseline Forecast Check - {cat}")
ax.set_xlabel("Date")
ax.set_ylabel("Units Sold")
ax.grid(False)
plt.show()

## 15. Save processed analytical tables

These files make the project reusable and avoid repeating expensive transformations.

In [ ]:
category_daily_raw.to_parquet(PROCESSED_DIR / "category_daily_raw.parquet", index=False)
category_daily_event_adjusted.to_parquet(PROCESSED_DIR / "category_daily_event_adjusted.parquet", index=False)
category_daily_normalized.to_parquet(PROCESSED_DIR / "category_daily_normalized_excluding_christmas.parquet", index=False)
store_category_daily.to_parquet(PROCESSED_DIR / "store_category_daily.parquet", index=False)
state_category_daily.to_parquet(PROCESSED_DIR / "state_category_daily.parquet", index=False)
series_profile.to_parquet(PROCESSED_DIR / "series_profile.parquet", index=False)

print("Saved processed tables to", PROCESSED_DIR)

## 16. EDA conclusions to update after running

After running the notebook, summarize the findings here:

1. **Data quality:** Are there duplicates, negative values, or missing values that require action?
2. **Hierarchy:** Do all stores carry all categories? Which categories/departments dominate?
3. **Christmas behavior:** How much does sales drop by category? Which stores/items still sell?
4. **Distribution:** Are category distributions distinct enough to require category-specific modeling?
5. **Seasonality:** Which categories have the strongest day-of-week or monthly patterns?
6. **Intermittency:** Which categories have the most sparse SKU-store series?
7. **Baseline forecasting:** Which simple baseline is hardest to beat?

Suggested next notebook: `01_feature_engineering.ipynb`